# Pipeline de pré-production

Le pipeline de pré-production enchaîne l'entraînement du modèle et le déploiement de l'API sur Cloud Run.
Le dépôt `purchase_predict_api` construit l'image, la pousse au registre et publie la révision.
Le dépôt `purchase_predict` lance ce build une fois `pytest` et `kedro run` terminés.
Un push sur la branche `staging` de l'API déclenche aussi ce build.
Le timeout couvre l'entraînement et le build imbriqué.


## Image et build de l'API

Le `Dockerfile` expose `$PORT`, lu au démarrage, et lance Gunicorn au premier plan. Le build a trois étapes : construction de l'image taguée `$SHORT_SHA`, push vers `gcr.io/$PROJECT_ID/purchase-predict-api`, déploiement Cloud Run en `europe-west1`.

Créer le déclencheur `build-purchase-predict-api-staging` sur la branche `staging` de `purchase_predict_api`. Autoriser le compte de service Cloud Build à déployer sur Cloud Run. Le champ `images` publie le même tag.


## Enchaînement depuis l'entraînement

Le `cloudbuild.yaml` de `purchase_predict` ajoute une étape `API Deploy`. Elle clone `purchase_predict_api` sur la branche `$BRANCH_NAME` avec le secret `GITHUB_TOKEN`, puis soumet le `cloudbuild.yaml` de l'API. `$$GITHUB_TOKEN` est réservé au shell, comme le secret du compte de service.

Créer le secret `GITHUB_TOKEN` (version `latest`) avec un jeton capable de cloner `github.com/<UTILISATEUR_GITHUB>/purchase_predict_api`. Le fichier conserve la machine `N1_HIGHCPU_8` et un timeout de 1200 s : le build imbriqué attend la fin du déploiement Cloud Run, au-delà de la limite par défaut de 10 minutes. Le paramètre Kedro `automl_max_evals` est fixé à 1 pour raccourcir l'essai.
